# SSTW Wan2.1 Flow Adapter Preflight - Colab 冷启动入口

该 Notebook 面向从空环境启动的 Colab。它会挂载 Google Drive、拉取仓库代码、安装依赖、下载 Wan2.1 权重、运行真实 GPU preflight, 并把结果打包保存到 Google Drive。

正式 records、artifacts 和 package manifest 均由仓库模块生成, Notebook 只作为入口。


## 阶段边界

本入口只验证: Wan2.1 能否加载、callback 是否捕获 latent、time grid 是否记录、sampler signature 是否记录、velocity / latent displacement proxy 是否可获得、L4 显存是否足够。

若 `adapter_preflight_decision != PASS`, 必须先修 adapter, 不得进入 B6 sampling-time constraint 或 full generative video probe。


In [ ]:
# 0. Notebook 进度显示 helper
# 该 helper 只用于 Colab 屏幕提示, 不写入正式 records、tables、figures、reports 或 claim artifacts。
import time
try:
    from IPython.display import HTML, display
except Exception:
    HTML = None
    display = None

SSTW_NOTEBOOK_PROGRESS_TOTAL = 11
SSTW_NOTEBOOK_PROGRESS_STARTED_AT = time.time()


def sstw_show_progress(step_index: int, step_name: str) -> None:
    """显示当前 Notebook 的阶段进度。"""
    elapsed_min = (time.time() - SSTW_NOTEBOOK_PROGRESS_STARTED_AT) / 60.0
    percent = 100.0 * float(step_index) / float(SSTW_NOTEBOOK_PROGRESS_TOTAL)
    message = (
        f"SSTW 进度: {step_index}/{SSTW_NOTEBOOK_PROGRESS_TOTAL} "
        f"({percent:.1f}%) | 已运行 {elapsed_min:.1f} min | {step_name}"
    )
    print(message)
    if HTML is not None and display is not None:
        try:
            display(HTML(
                "<div style='border:1px solid #bbb;padding:8px;border-radius:6px;'>"
                f"<div><b>SSTW Notebook 进度</b>: {step_index}/{SSTW_NOTEBOOK_PROGRESS_TOTAL} "
                f"({percent:.1f}%)</div>"
                "<div style='background:#eee;height:12px;border-radius:8px;overflow:hidden;margin-top:6px;'>"
                f"<div style='background:#2f80ed;width:{percent:.1f}%;height:12px;'></div>"
                "</div>"
                f"<div style='margin-top:6px;'>当前阶段: {step_name}</div>"
                f"<div style='margin-top:2px;color:#555;'>已运行: {elapsed_min:.1f} min</div>"
                "</div>"
            ))
        except Exception:
            pass


sstw_show_progress(1, '1. 挂载 Google Drive 并检查 GPU')

# 1. 挂载 Google Drive 并检查 GPU
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess, sys, json
from pathlib import Path

print(subprocess.getoutput('nvidia-smi'))
DRIVE_PROJECT_ROOT = '/content/drive/MyDrive/SSTW'
Path(DRIVE_PROJECT_ROOT).mkdir(parents=True, exist_ok=True)
print('drive_project_root =', DRIVE_PROJECT_ROOT)


In [ ]:
sstw_show_progress(2, '2. 冷启动获取仓库代码')
# 2. 冷启动获取仓库代码
# 如果使用私有 fork, 请把 REPO_URL 改成你自己的可访问地址。
REPO_URL = 'https://github.com/RICHAAARC/SSTW.git'
REPO_DIR = '/content/SSTW'

if not Path(REPO_DIR).exists():
    if not REPO_URL:
        raise RuntimeError('请先设置 REPO_URL, 或将仓库上传到 /content/SSTW')
    !git clone "$REPO_URL" "$REPO_DIR"
else:
    print('仓库目录已存在, 执行 git pull 以同步远端代码。')
    %cd "$REPO_DIR"
    !git pull --ff-only
%cd "$REPO_DIR"


In [ ]:
sstw_show_progress(3, '3. 安装 Colab GPU 运行依赖')
# 3. 安装 Colab GPU 运行依赖
# 使用 diffusers GitHub 版本, 以提高 WanPipeline 可用概率。
%pip install -U git+https://github.com/huggingface/diffusers transformers accelerate safetensors imageio imageio-ffmpeg sentencepiece protobuf huggingface_hub


In [ ]:
sstw_show_progress(4, '4. Hugging Face 认证')
# 4. Hugging Face 认证
# 推荐在 Colab Secrets 中配置 HF_TOKEN。此处只读取环境变量, 不打印 token 值, 不写入 records。
import os
from huggingface_hub import login

if os.environ.get('HF_TOKEN'):
    login(token=os.environ['HF_TOKEN'], add_to_git_credential=False)
    print('HF_TOKEN 状态: provided from environment.')
else:
    print('HF_TOKEN 状态: not_provided in environment, 将尝试匿名下载公开模型。')


In [ ]:
sstw_show_progress(5, '5. 初始化 Drive 目录布局')
# 5. 初始化 Drive 目录布局
from paper_workflow.notebook_utils import flow_model_adapter_preflight_workflow as preflight_workflow

layout = preflight_workflow.ensure_drive_layout(DRIVE_PROJECT_ROOT)
print(json.dumps(layout, ensure_ascii=False, indent=2))


In [ ]:
sstw_show_progress(6, '6. 先运行默认测试和 harness 审计, 确认 Colab 中仓库状态满足治理约束。')
# 6. 先运行默认测试和 harness 审计, 确认 Colab 中仓库状态满足治理约束。
!pytest -q
!python tools/harness/run_all_audits.py


In [ ]:
sstw_show_progress(7, '7. 配置真实 Wan2.1 GPU preflight')
# 7. 配置真实 Wan2.1 GPU preflight
MODEL_ID = preflight_workflow.DEFAULT_WAN21_PREFLIGHT_MODEL_ID
NUM_INFERENCE_STEPS = 4
NUM_FRAMES = 33
HEIGHT = 320
WIDTH = 512

print('MODEL_ID =', MODEL_ID)
print('preflight output =', layout['drive_run_root'])


In [ ]:
sstw_show_progress(8, '8. 执行真实 GPU preflight')
# 8. 执行真实 GPU preflight
# 正式 records / artifacts 会由 experiments 模块写入 Drive run 子目录。
cmd = preflight_workflow.build_wan21_flow_adapter_preflight_command(
    layout,
    model_id=MODEL_ID,
    num_inference_steps=NUM_INFERENCE_STEPS,
    num_frames=NUM_FRAMES,
    height=HEIGHT,
    width=WIDTH,
)
print(' '.join(cmd))
result = preflight_workflow.run_command(cmd)
print(result.stdout)
print(result.stderr)
if result.returncode != 0:
    raise SystemExit(result.returncode)


In [ ]:
sstw_show_progress(9, '9. 检查 preflight decision。失败时停止, 不进入 B6。')
# 9. 检查 preflight decision。失败时停止, 不进入 B6。
decision_path = Path(layout['drive_run_root']) / 'artifacts' / 'wan21_flow_adapter_preflight_decision.json'
decision = json.loads(decision_path.read_text(encoding='utf-8'))
print(json.dumps(decision, ensure_ascii=False, indent=2, sort_keys=True))

assert decision['adapter_preflight_decision'] == 'PASS', 'preflight 未通过, 不得进入 B6 sampling-time constraint'
assert decision['model_load_status'] == 'loaded'
assert decision['callback_latent_capture_status'] == 'captured'
assert decision['time_grid_capture_status'] == 'captured'
assert decision['sampler_signature_status'] == 'captured'
assert decision['velocity_proxy_status'] == 'captured'


In [ ]:
sstw_show_progress(10, '10. 打包到 Google Drive packages/wan21_flow_adapter_preflight')
# 10. 打包到 Google Drive packages/wan21_flow_adapter_preflight
# 打包逻辑由仓库脚本执行, Notebook 不直接创建 package manifest。
cmd = preflight_workflow.build_drive_packaging_command(layout)
print(' '.join(cmd))
result = preflight_workflow.run_command(cmd)
print(result.stdout)
print(result.stderr)
if result.returncode != 0:
    raise SystemExit(result.returncode)


In [ ]:
sstw_show_progress(11, '11. 显示可下载审阅文件。Colab 断开后, 这些文件仍保存在 Google Drive。')
# 11. 显示可下载审阅文件。Colab 断开后, 这些文件仍保存在 Google Drive。
package_dir = Path(layout['drive_package_dir'])
print('package_dir =', package_dir)
for path in sorted(package_dir.glob('*')):
    print(path)
